In [1]:
import torch 
import lightning as L
import yaml
import sys, os
sys.path.append('../')
# sys.path.append('lightning_scripts/')
from lightning_scripts.lightning_ssl import LitAudioSSL 
import importlib
from pathlib import Path 
# from lightning_scripts.jsinV3DataLoader_precombined_batched import jsinV3_precombined_all_signals



In [20]:
# get config and init trained model 
config_path = Path("model_configs/mmcr_search/ssl_mmcr_word_resnet50_hparam_set_0.yaml")
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 4
config['hparas']['batch_size'] = 96 # set to single-gpu size 

# update val set to use entire range 
config['data']['eval_max'] = 1

checkpoint_dir = Path("model_checkpoints") / f"{config_path.stem}/checkpoints"
ckpt_paths = sorted(checkpoint_dir.glob("*.ckpt"), key=os.path.getctime)
ckpt_path = ckpt_paths[-1] # get latest checkpoint 
print(ckpt_path)

model_checkpoints/ssl_mmcr_word_resnet50_hparam_set_0/checkpoints/epoch=4-step=56500.ckpt


In [21]:
model = LitAudioSSL.load_from_checkpoint(checkpoint_path=ckpt_path, config=config).eval()

In [22]:
config['data']

{'root': '/mnt/ceph/users/jfeather/data/training_datasets_audio/JSIN_all_v3/subsets/',
 'eval_max': 1}

In [23]:
## init val dataloader

val_loader = model.val_dataloader() # will be populated with relevant args in config 

In [24]:
trainer = L.Trainer(devices=1)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [25]:
trainer.test(model, dataloaders=val_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  test_class_acc_epoch     0.026400862261652946
  test_class_loss_epoch      6.330034255981445
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_class_acc_epoch': 0.026400862261652946,
  'test_class_loss_epoch': 6.330034255981445}]